In [13]:
import pandas, numpy, sklearn, xgboost
print(pandas.__version__, numpy.__version__, sklearn.__version__, xgboost.__version__)

2.2.3 1.24.3 1.6.1 2.1.4


In [14]:
import pandas as pd


## MESSAGE FOR TEAMMATES ==> CHANGE THIS ON OWN MACHINE 
base_path = '~/Desktop/SCHOOL/SEM2/Advanced AI/BUSit week/Attendance AI/Project 2 - Matchday Attendance Prediction/Data/'

df_match = pd.read_csv(base_path + 'gold_match.csv')
df_trends = pd.read_csv(base_path + 'gold_google_trends_daily.csv')
df_tickets = pd.read_csv(base_path + 'gold_match_tickets.csv')
df_context = pd.read_csv(base_path + 'gold_match_context.csv')
df_goals = pd.read_csv(base_path + 'gold_match_goals.csv')
df_articles = pd.read_csv(base_path + 'gold_belga_press_articles.csv', 
                          escapechar='\\', on_bad_lines='skip')


Imports

In [15]:
import pandas as pd
import numpy as np
import pickle, json

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut, cross_val_score

try:
    from xgboost import XGBRegressor
    use_xgb = True
except ImportError:
    use_xgb = False

import warnings
warnings.filterwarnings('ignore')

OUTPUT_PATH = "/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/BUSit week/Attendance AI/MODEL/OHL-Ai-project/data/outputs/"

DATA PREP

In [16]:
# ── Base: home matches only ──────────────────────────────────────────────────
df = df_match[df_match['is_home_match'] == True].copy()
df = df.merge(df_tickets, on='match_id', how='left')
df = df.merge(df_context, on='match_id', how='left')
df['match_date'] = df['match_date_x']
df = df.drop(columns=['match_date_x', 'match_date_y'], errors='ignore')

# Trends & articles
df_trends_agg = df_trends.groupby('match_id')['ohl_interest'].mean().reset_index()
df = df.merge(df_trends_agg, on='match_id', how='left')

df_articles_agg = df_articles.groupby('match_id').size().reset_index(name='article_count')
df = df.merge(df_articles_agg, on='match_id', how='left')
df['article_count'] = df['article_count'].fillna(0)

df = df.sort_values('match_date').reset_index(drop=True)

Feature Engineering (Combined)


In [17]:
df['kickoff_hour'] = pd.to_datetime(
    df['kickoff_time_local'], format='%H:%M:%S'
).dt.hour

# ── Form features (Model 2) ──────────────────────────────────────────────────
points_map = {'W': 3, 'D': 1, 'L': 0}
df['points']           = df['result_home'].map(points_map)
df['points_last_5']    = df['points'].rolling(5).sum().shift(1).fillna(df['points'].mean() * 5)
df['win']              = (df['result_home'] == 'W').astype(int)
df['wins_last_3']      = df['win'].rolling(3).sum().shift(1).fillna(df['win'].mean() * 3)
df['goal_diff']        = df['goals_home_ft'] - df['goals_away_ft']
df['goal_diff_last_5'] = df['goal_diff'].rolling(5).sum().shift(1).fillna(0)

# ── Season & matchday features ───────────────────────────────────────────────
df['matchday']        = pd.to_numeric(df['matchday'], errors='coerce')
df['season_progress'] = df['matchday'] / df['matchday'].max()

# ── Match importance composite ───────────────────────────────────────────────
df['form_strength_norm'] = (
    (df['points_last_5'] - df['points_last_5'].min()) /
    (df['points_last_5'].max() - df['points_last_5'].min())
)
df['match_importance']   = 0.5 * df['form_strength_norm'] + 0.5 * df['season_progress']
df['is_high_importance'] = (df['match_importance'] > df['match_importance'].median()).astype(int)

# ── Opponent features (Model 1 + 2 combined) ────────────────────────────────
df['opponent']      = df['away_team']
df['opponent_freq'] = df['opponent'].map(df['opponent'].value_counts())

top_teams = ["Club Brugge", "Anderlecht", "STVV", "KV Mechelen", "Westerlo"]
df['is_top_opponent'] = df['away_team'].isin(top_teams).astype(int)

df['opponent_strength_norm'] = (
    (df['opponent_freq'] - df['opponent_freq'].min()) /
    (df['opponent_freq'].max() - df['opponent_freq'].min())
)

# ── Match attractiveness (Model 2) ───────────────────────────────────────────
df['match_attractiveness'] = (
    0.4 * df['match_importance'] +
    0.4 * df['opponent_strength_norm'] +
    0.2 * df['is_top_opponent']
)
df['form_x_opponent'] = df['points_last_5'] * df['is_top_opponent']

# ── Opponent avg attendance (Model 1) ────────────────────────────────────────
df['opponent_avg_attendance_raw'] = df.groupby('away_team')['tickets_sold_total'].transform('mean')

# ── Lag & promo ──────────────────────────────────────────────────────────────
df['attendance_lag_1'] = df['tickets_scanned'].shift(1).fillna(df['tickets_scanned'].mean())
df['has_promotion']    = df['has_promotion'].astype(int)

print("Feature engineering complete:", df.shape)

Feature engineering complete: (71, 90)


Sigmoid Normalization (Optional)


In [18]:
STADIUM_CAPACITY = 10_000  # Den Dreef — adjust if needed
USE_SIGMOID_TARGET = False  # set True to experiment

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def normalize_attendance(y, capacity=STADIUM_CAPACITY):
    """Scale to 0-1 then apply sigmoid to compress extremes."""
    rate = y / capacity          # attendance rate
    centered = rate - 0.5        # center around 0
    return sigmoid(centered * 6) # stretch so range is meaningful

def inverse_sigmoid_attendance(y_sig, capacity=STADIUM_CAPACITY):
    """Reverse the sigmoid normalization back to ticket count."""
    clipped = np.clip(y_sig, 1e-6, 1 - 1e-6)
    centered = np.log(clipped / (1 - clipped)) / 6
    return (centered + 0.5) * capacity

Features set

In [ ]:
features_combined = [
    
    'opponent_avg_attendance_raw',
    'academic_week',
    'matchday',
    'weather_rain_mm',
    'kickoff_hour',
    
    'points_last_5',
    'wins_last_3',
    'goal_diff_last_5',
    
    'match_importance',
    'match_attractiveness',
    'form_x_opponent',
    
    'is_weekend',
    'season_progress',
    'has_promotion',
    'attendance_lag_1',
]

target = 'tickets_scanned'

df_model = df[features_combined + [target, 'away_team']].dropna()
X = df_model[features_combined]
y = df_model[target]

if USE_SIGMOID_TARGET:
    y_fit = normalize_attendance(y)
else:
    y_fit = y

print(f"Final dataset: {df_model.shape}")

# ── Models ───────────────────────────────────────────────────────────────────
models = {
    "LinearRegression": LinearRegression(),
    "Ridge":            Ridge(alpha=10),
    "RandomForest":     RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42),
}
if use_xgb:
    models["XGBoost"] = XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)

# ── LOOCV ────────────────────────────────────────────────────────────────────
loo = LeaveOneOut()
results, predictions_dict = {}, {}

for name, model in models.items():
    y_true_list, y_pred_list = [], []

    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train_fold    = y_fit.iloc[train_idx]

        # Leak-free opponent avg: computed only on train fold
        opp_avg = df_model.iloc[train_idx].groupby('away_team')[target].mean()
        global_avg_fold = y.iloc[train_idx].mean()
        opp_test = df_model.iloc[test_idx]['away_team'].values[0]
        X_train = X_train.copy()
        X_test  = X_test.copy()
        X_train['opponent_avg_attendance_raw'] = df_model.iloc[train_idx]['away_team'].map(opp_avg)
        X_test['opponent_avg_attendance_raw']  = opp_avg.get(opp_test, global_avg_fold)

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_train)
        X_te_s = scaler.transform(X_test)

        model.fit(X_tr_s, y_train_fold)
        pred = model.predict(X_te_s)[0]

        if USE_SIGMOID_TARGET:
            pred = inverse_sigmoid_attendance(pred)

        y_true_list.append(y.iloc[test_idx].values[0])
        y_pred_list.append(pred)

    results[name] = {
        "MAE":  mean_absolute_error(y_true_list, y_pred_list),
        "RMSE": np.sqrt(mean_squared_error(y_true_list, y_pred_list)),
        "R2":   r2_score(y_true_list, y_pred_list),
        "MAPE": np.mean(np.abs((np.array(y_true_list) - np.array(y_pred_list)) / np.array(y_true_list))) * 100
    }
    predictions_dict[name] = pd.DataFrame({"Actual": y_true_list, "Predicted": y_pred_list})

results_df = pd.DataFrame(results).T.sort_values("MAE")
print(results_df.round(2))

Final dataset: (71, 17)
                      MAE     RMSE    R2   MAPE
RandomForest      1200.95  1460.04  0.46  19.81
GradientBoosting  1281.76  1570.51  0.37  20.58
XGBoost           1304.91  1709.57  0.26  21.79
Ridge             1321.71  1604.77  0.34  21.05
LinearRegression  1349.66  1655.79  0.30  21.46


SAVE BEST MODEL


In [23]:
best_model_name = results_df.index[0]
print(f"\n✅ Best model: {best_model_name}")

# Retrain best model on full data
scaler_final = StandardScaler()
X_scaled_final = scaler_final.fit_transform(X)
best_model = models[best_model_name]
best_model.fit(X_scaled_final, y_fit)

std_resid = (y_fit - pd.Series(best_model.predict(X_scaled_final))).std()

with open('/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/BUSit week/Attendance AI/MODEL/OHL-Ai-project/notebooks/interactive_interface/model.pkl', 'wb') as f:
    pickle.dump({
        'model':    best_model,
        'scaler':   scaler_final,
        'std':      float(std_resid),
        'features': features_combined,
        'sigmoid':  USE_SIGMOID_TARGET,
        'capacity': STADIUM_CAPACITY
    }, f)

opp_lookup = df_model.groupby('away_team')[target].mean().round(0).to_dict()
opp_lookup['__global_avg__'] = float(y.mean())
with open('/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/BUSit week/Attendance AI/MODEL/OHL-Ai-project/notebooks/interactive_interface/opponent_lookup.json', 'w') as f:
    json.dump(opp_lookup, f, indent=2)

results_df.to_csv(OUTPUT_PATH + "model_results_combined.csv")
print("✅ model.pkl, opponent_lookup.json, model_results_combined.csv saved.")


✅ Best model: RandomForest
✅ model.pkl, opponent_lookup.json, model_results_combined.csv saved.
